# 01 — Initial Inspection

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
pd.set_option('display.max_columns', 50)

from src.data.load_data import load_csv
from src.utils.config import load_config, resolve_path

config = load_config()
raw_path = resolve_path(config['data']['raw_path'])
df = load_csv(raw_path)
print(f"Dataset loaded from: {raw_path}")

2026-09-02 01:17:40 | INFO     | src.data.load_data | Loaded CSV 'manufacturing_defect_dataset.csv' with shape (3240, 17)


Dataset loaded from: /home/claude/work/project/manufacturing-quality-operational-performance-analysis/data/raw/manufacturing_defect_dataset.csv


## Dataset shape


In [2]:
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")

Shape: 3240 rows, 17 columns


**Interpretation:** The dataset contains 3,240 individual production records, each described by 17 variables. This is a single flat table — there is no separate file for suppliers, equipment, or dates, so all analysis in this project must work within this one table.

## Column names


In [3]:
list(df.columns)

['ProductionVolume',
 'ProductionCost',
 'SupplierQuality',
 'DeliveryDelay',
 'DefectRate',
 'QualityScore',
 'MaintenanceHours',
 'DowntimePercentage',
 'InventoryTurnover',
 'StockoutRate',
 'WorkerProductivity',
 'SafetyIncidents',
 'EnergyConsumption',
 'EnergyEfficiency',
 'AdditiveProcessTime',
 'AdditiveMaterialCost',
 'DefectStatus']

## Data types



In [4]:
df.dtypes

ProductionVolume          int64
ProductionCost          float64
SupplierQuality         float64
DeliveryDelay             int64
DefectRate              float64
QualityScore            float64
MaintenanceHours          int64
DowntimePercentage      float64
InventoryTurnover       float64
StockoutRate            float64
WorkerProductivity      float64
SafetyIncidents           int64
EnergyConsumption       float64
EnergyEfficiency        float64
AdditiveProcessTime     float64
AdditiveMaterialCost    float64
DefectStatus              int64
dtype: object

**Interpretation:** All 17 columns are numeric (`int64` or `float64`). There are no text/categorical columns and, importantly, **no date column** — this immediately tells us that time-trend analysis will not be possible with this dataset.

## First look at the raw data


In [5]:
df.head(10)

,ProductionVolume,ProductionCost,SupplierQuality,DeliveryDelay,DefectRate,QualityScore,MaintenanceHours,DowntimePercentage,InventoryTurnover,StockoutRate,WorkerProductivity,SafetyIncidents,EnergyConsumption,EnergyEfficiency,AdditiveProcessTime,AdditiveMaterialCost,DefectStatus
0,202,13175.403783,86.648534,1,3.121492,63.463494,9,0.052343,8.630515,0.081322,85.042379,0,2419.616785,0.468947,5.551639,236.439301,1
1,535,19770.046093,86.310664,4,0.819531,83.697818,20,4.908328,9.296598,0.038486,99.657443,7,3915.566713,0.119485,9.080754,353.957631,1
2,960,19060.820997,82.132472,0,4.514504,90.350550,1,2.464923,5.097486,0.002887,92.819264,2,3392.385362,0.496392,6.562827,396.189402,1
3,370,5647.606037,87.335966,5,0.638524,67.628690,8,4.692476,3.577616,0.055331,96.887013,8,4652.400275,0.183125,8.097496,164.135870,1
4,206,7472.222236,81.989893,3,3.867784,82.728334,9,2.746726,6.851709,0.068047,88.315554,7,1581.630332,0.263507,6.406154,365.708964,1
5,171,6975.931602,95.331919,1,3.914574,92.568436,19,3.027324,7.930009,0.074069,87.079118,7,1238.994421,0.118021,7.279442,171.711804,1
6,800,15889.698650,99.325486,3,4.789000,90.729911,10,3.559561,3.046889,0.040192,91.063158,8,3138.431150,0.333913,4.891669,188.727737,1
7,120,17266.779948,99.401489,4,0.743605,92.119681,13,1.604879,8.380972,0.009702,88.705569,3,1004.108554,0.293422,9.333835,312.526896,1
8,714,8202.670495,97.301422,5,3.185856,95.172937,2,3.494920,3.668747,0.058433,94.298961,4,4150.875773,0.366683,5.517451,215.680921,1
9,221,12587.790394,92.015843,2,2.425283,97.507284,0,2.633960,5.933418,0.032955,85.316362,6,3023.891555,0.317071,5.965972,364.638176,0


## Missing values check


In [6]:
total_missing = df.isnull().sum().sum()
print(f"Total missing values across the entire dataset: {total_missing}")
df.isnull().sum()

Total missing values across the entire dataset: 0


ProductionVolume        0
ProductionCost          0
SupplierQuality         0
DeliveryDelay           0
DefectRate              0
QualityScore            0
MaintenanceHours        0
DowntimePercentage      0
InventoryTurnover       0
StockoutRate            0
WorkerProductivity      0
SafetyIncidents         0
EnergyConsumption       0
EnergyEfficiency        0
AdditiveProcessTime     0
AdditiveMaterialCost    0
DefectStatus            0
dtype: int64

**Interpretation:** Zero missing values were found in any column. This is an unusually clean dataset — no imputation strategy will be required in the cleaning stage (see `notebooks/03_data_cleaning.ipynb`).

## Duplicate rows check


In [7]:
dup_count = df.duplicated().sum()
print(f"Duplicate rows: {dup_count}")

Duplicate rows: 0


**Interpretation:** No duplicate rows were found. Every one of the 3,240 records is unique across all 17 columns.

## Identifying column categories


In [8]:
column_categories = {
    'Production': ['ProductionVolume', 'ProductionCost'],
    'Supplier': ['SupplierQuality', 'DeliveryDelay'],
    'Quality / Defect': ['DefectRate', 'QualityScore', 'DefectStatus'],
    'Maintenance / Downtime': ['MaintenanceHours', 'DowntimePercentage'],
    'Inventory': ['InventoryTurnover', 'StockoutRate'],
    'Workforce / Safety': ['WorkerProductivity', 'SafetyIncidents'],
    'Energy': ['EnergyConsumption', 'EnergyEfficiency'],
    'Additive Process': ['AdditiveProcessTime', 'AdditiveMaterialCost'],
}
for category, cols in column_categories.items():
    print(f"{category}: {cols}")

Production: ['ProductionVolume', 'ProductionCost']
Supplier: ['SupplierQuality', 'DeliveryDelay']
Quality / Defect: ['DefectRate', 'QualityScore', 'DefectStatus']
Maintenance / Downtime: ['MaintenanceHours', 'DowntimePercentage']
Inventory: ['InventoryTurnover', 'StockoutRate']
Workforce / Safety: ['WorkerProductivity', 'SafetyIncidents']
Energy: ['EnergyConsumption', 'EnergyEfficiency']
Additive Process: ['AdditiveProcessTime', 'AdditiveMaterialCost']


## Dataset summary and limitations

**Summary of this dataset:**
- 3,240 production records, 17 numeric columns, no missing values, no duplicates.
- One binary target column: `DefectStatus` (1 = defective, 0 = not defective).
- Covers production, cost, supplier quality, quality/defect, maintenance, downtime, inventory, workforce, safety, energy, and additive-process variables.

**Confirmed limitations (will be documented throughout the project):**
- No date/time column -> no time-trend analysis is possible.
- No supplier ID, product ID, plant ID, or equipment ID -> no entity-level comparisons are possible, only continuous-variable-level analysis (e.g. by supplier-quality *score range*, not by named supplier).
- Single flat table -> no joins are needed or possible; all analysis is self-contained within this one dataset.


